# MAAP Docs-Style OGCification Sample Notebook

This notebook is intentionally structured like MAAP documentation tutorials: overview, setup, parameters, data discovery, data access, processing, output writing, optional visualization, and DPS/OGC readiness notes.

It is designed for testing the `NISAR_OGC_JOB` generator pipeline. The notebook does **not** overwrite or depend on real MAAP catalog defaults. Any MAAP/STAC/CMR values should be supplied by the user, live metadata lookup, or MCP default-resolution output.


## 1. What this notebook demonstrates

- Parameter values collected in a Papermill-style `parameters` cell.
- Optional MAAP/STAC metadata context.
- A DPS-ready processing function.
- Output writing under `output/`, matching OGC/MAAP expectations.
- An optional visualization section that should remain notebook-only.


In [ ]:
# Runtime/catalog parameters. Leave blank values empty unless metadata lookup or the user fills them.
short_name = ""
collection_id = ""
concept_id = ""
version = ""
asset_href = ""
asset_key = ""
bbox = ""
datetime = ""
crs = "EPSG:4326"
group = ""
variables = "synthetic_band"
access_mode = "auto"
example_input_file = ""
output_dir = "output"
maap_stac_browser_url = "https://stac-browser.maap-project.org/?.language=en"


## 2. Setup

MAAP tutorials often initialize `maap-py` in setup cells. This sample keeps that optional so the notebook can run locally as well as inside MAAP ADE.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

try:
    from maap.maap import MAAP
except ImportError:
    MAAP = None

try:
    import numpy as np
except ImportError as exc:
    raise RuntimeError("This sample notebook requires numpy for the synthetic fallback run.") from exc


## 3. Data discovery

In a real MAAP workflow, this section would use MAAP, STAC, or CMR metadata to resolve values such as `short_name`, `collection_id`, `asset_href`, `bbox`, and `datetime`. For this sample, unresolved values are recorded as metadata and synthetic data is used for a safe local run.


In [ ]:
def build_discovery_context():
    """Collect user/MCP/catalog parameters into one metadata object."""
    return {
        "short_name": short_name,
        "collection_id": collection_id,
        "concept_id": concept_id,
        "version": version,
        "asset_href": asset_href,
        "asset_key": asset_key,
        "bbox": bbox,
        "datetime": datetime,
        "crs": crs,
        "group": group,
        "variables": variables,
        "access_mode": access_mode,
        "maap_stac_browser_url": maap_stac_browser_url,
    }


discovery_context = build_discovery_context()
discovery_context


## 4. Data access

This cell is DPS-candidate logic. If `asset_href` or `example_input_file` is provided, package generation can treat those as runtime inputs. If neither exists, the sample uses deterministic synthetic data.


In [ ]:
def load_or_create_input(context):
    """Return small deterministic input data for local smoke tests and DPS packaging demos."""
    if example_input_file:
        path = Path(example_input_file)
        if not path.exists():
            raise FileNotFoundError(f"example_input_file does not exist: {path}")
        return np.loadtxt(path, delimiter=",")

    # Safe local fallback: no catalog values are hardcoded here.
    rows = np.arange(25, dtype=float).reshape(5, 5)
    return rows


input_array = load_or_create_input(discovery_context)
input_array.shape


## 5. Processing / subsetting

This is the main DPS-ready science function. It receives data and parameters, performs deterministic processing, and returns serializable results.


In [ ]:
def process_subset(array, context):
    """Compute a tiny deterministic summary that stands in for a real remote-sensing subset."""
    selected_variables = [item.strip() for item in str(context.get("variables", "")).split(",") if item.strip()]
    return {
        "shape": list(array.shape),
        "mean": float(array.mean()),
        "min": float(array.min()),
        "max": float(array.max()),
        "variables": selected_variables,
        "bbox": context.get("bbox", ""),
        "crs": context.get("crs", ""),
    }


result = process_subset(input_array, discovery_context)
result


## 6. Output writing

DPS/OGC jobs should write outputs under `output/` or the configured output directory. The generator and validator check for this pattern.


In [ ]:
def write_outputs(result, context, output_dir=output_dir):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    summary_path = output_path / "summary.json"
    provenance_path = output_path / "provenance.json"

    summary_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    provenance_path.write_text(json.dumps(context, indent=2) + "\n", encoding="utf-8")
    return {"summary": str(summary_path), "provenance": str(provenance_path)}


outputs = write_outputs(result, discovery_context)
outputs


## 7. Optional visualization

This cell is useful in a notebook but should not be required for DPS execution. The DPS-readiness scanner should classify this as notebook-only or optional visualization.


In [ ]:
try:
    import matplotlib.pyplot as plt

    plt.imshow(input_array)
    plt.title("Synthetic input preview")
    plt.colorbar()
    plt.show()
except ImportError:
    print("matplotlib is not installed; skipping optional visualization")


## 8. DPS / OGC readiness notes

- DPS-ready sections: parameters, discovery context, data access, processing, output writing.
- Notebook-only section: optional visualization.
- Values such as `short_name`, `collection_id`, `asset_href`, `bbox`, and `datetime` are intentionally parameters, not hardcoded defaults.
- Run the generator with `--emit-suggested-notebook-v2` to create a non-destructive cleaner notebook draft.
